In [1]:
import spacy
from FlagEmbedding import FlagModel
from elasticsearch import Elasticsearch
from elasticsearch import helpers

In [2]:
# IndexText uses an embedding model to generate vector embeddings of an input text and stores these embeddings in a 
# vector database using Elasticsearch. The input text in split into chunks of multiple sentences, then embedded and indexed.
# The vector database takes the form (text_chunk_i, vector_embedding_i) for the ith entry of the index.

In [3]:
# helper functions
def remove_newline(text):
# removes newline characters, "\n", from text
# text: list of paragraphs in the text
    
    for i in range(len(text)):
        text[i] = " ".join(text[i].split())

    return text

def embed_index_text(text_chunks, client):
    # chunks: list of m elements that contain n sentences each
    # client: instance of Elasticsearch client used to create the index

    # embed chunks
    chunk_embeddings = model.encode(text_chunks).tolist()

    # define the format of the data to be indexed as pairs (chunk of text, chunk embeddings)
    docs = [
        {
            '_op_type': 'index',
            '_index': 'text_embeddings_index',
            '_source': {
                "chunk" : t, 
                "embedding_vector" : v
            }
        } for t, v in zip(text_chunks, chunk_embeddings)
    ]
    
    # index in bulk
    res = helpers.bulk(client, docs)
    # print(res)

def createIndex(client, index_name):
    # client: instance of Elasticsearch client
    # index_name (str): index name

    # ensure that there is no previously defined index under the index_name
    if (client.indices.exists(index = index_name)):
        client.indices.delete(index = index_name)

    # define the format of index: chunk of text and embedding vectors
    # custom mapping that defines the expected types of indices features
    # define mapping parameters for the "chunk" and "embedding_vector" fields
    # define "vector_dim"
    mappings = {
        "properties": {
            "chunk": {
                "type": "text"
            }, 
            "embedding_vector": {
                "index": True, 
                "type": "dense_vector", 
                "dims": 512, 
                "similarity": "cosine",
            }
        }
    }
        
    # create index
    client.indices.create(index = index_name, mappings = mappings)


In [25]:
class IndexText:

    def __init__(self, embedding_model):
        self.embedding_model = FlagModel('BAAI/bge-small-zh-v1.5', use_fp16 = True)

    def __preProcessInput(self, file_path):
        # text: text file

        with(open(file_path, "r")) as text_file:
            text = text_file.read()

        # split text in paragraphs
        text = text.split("\n\n")

        # using the helper function "remove_newline" to eliminate "\n" characters from the text
        text = remove_newline(text)

        return text

    def chunkEmbedIndex(self, file_path):
        text = self.__preProcessInput(file_path)
        
        # Load pretrained English Language Model to separate the text into sentences
        nlp = spacy.load('en_core_web_sm') 
        
        doc_pipeline = nlp.pipe(text, batch_size = 5, n_process = 1)




In [26]:
model = FlagModel('BAAI/bge-small-zh-v1.5', use_fp16 = True)

In [27]:
les_miserables = IndexText(model)

In [28]:
text = les_miserables.preProcessInput("../data/LesMiserables.txt")

In [29]:
len(text)

13942

In [36]:
# instantiate Python client for Elasticsearch
client = Elasticsearch("http://elasticsearch:9200")

In [37]:
client.indices.exists(index = "test_index")

HeadApiResponse(True)